# U19 — Data Labeling & Data-Centric AI: Lab

### Real-world brief: cleaning noisy inspection labels on an aluminium casting line

A foundry classifies castings as **defect** or **ok**. Labels come from **human inspectors who disagree**, especially on borderline parts — so the labels you train on contain errors. In the *data-centric* philosophy, improving these labels often beats swapping in a fancier model. In this lab you'll measure **inter-annotator agreement**, build **consensus labels**, **detect likely label errors**, run an **active-learning** loop, and try **weak supervision** — then show that fixing the data lifts performance.

**Resource provided:** `casting_inspection.csv` — objective measurements, three inspector labels (`inspector_A/B/C`), the noisy `label_recorded` you'd normally train on, and a hidden `true_defect` ground truth (provided here only so you can *measure* progress). Keep it beside this notebook.

_Phase E — Data-Centric AI._

#objectives

Quantify inter-annotator agreement with Cohen's kappa

Build consensus (majority-vote) labels and compare to ground truth

Detect likely mislabelled rows with confident-learning-style logic

Show that cleaning labels improves a model (data-centric win)

Run an active-learning loop and combine weak-supervision rules

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [1]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_castings(csv_path="casting_inspection.csv", seed=190, verbose=False):
    """Aluminium casting inspection records for a DATA-CENTRIC AI lab (U19).

    Each casting has objective process/measurement features and a true (latent) defect
    state. Three human inspectors each label it — but humans are noisy and disagree, more
    so on borderline parts. The column you'd actually TRAIN on (`label_recorded`) is a
    single inspector's call and therefore contains real labelling errors.

    Columns:
      porosity_pct, wall_thickness_mm, fill_time_s, melt_temp_c, pressure_bar, surface_ra_um
                                          -> objective features
      inspector_A / inspector_B / inspector_C  -> three human labels (0 ok / 1 defect)
      label_recorded                      -> the noisy single-annotator label (train on this)
      true_defect                         -> hidden ground truth (for teaching/validation only)
    """
    rng = np.random.default_rng(seed)
    N = 2400

    porosity = rng.gamma(2.0, 1.1, N).clip(0, 12)
    wall = rng.normal(6.0, 0.8, N).clip(3.5, 8.5)
    fill_time = rng.normal(2.4, 0.5, N).clip(1.0, 4.5)
    melt_temp = rng.normal(710, 15, N).clip(660, 760)
    pressure = rng.normal(95, 12, N).clip(60, 130)
    surface_ra = rng.normal(3.2, 0.9, N).clip(1.0, 7.0)

    # true defect: a SHARP function of genuine drivers so it is learnable from features
    # (steep sigmoid -> probabilities pushed toward 0/1 -> high but not perfect ceiling)
    drive = (0.55 * porosity + 1.3 * np.maximum(4.8 - wall, 0)
             + 1.1 * np.maximum(fill_time - 2.9, 0) + 0.45 * np.maximum(surface_ra - 3.6, 0)
             + 0.04 * np.maximum(melt_temp - 720, 0))
    thr = np.quantile(drive, 0.80)            # ~20% defect rate
    p_true = 1 / (1 + np.exp(-2.2 * (drive - thr)))   # gain 2.2 -> sharp, ~10% label noise
    true_defect = (rng.random(N) < p_true).astype(int)

    # "difficulty": borderline parts (p near 0.5) are where inspectors disagree
    difficulty = 1 - np.abs(p_true - 0.5) * 2          # 0 easy .. 1 hard
    def inspector(skill):
        # flip the true label with prob rising on hard parts, lower for higher skill
        flip_p = (0.07 + 0.55 * difficulty) * (1.0 - skill)
        flips = rng.random(N) < flip_p
        return np.where(flips, 1 - true_defect, true_defect)

    insp_A = inspector(0.80)
    insp_B = inspector(0.66)
    insp_C = inspector(0.52)          # least reliable inspector

    # label_recorded = the noisy HISTORICAL label. It carries a real inspector VISUAL BIAS:
    # rough-looking but truly-OK parts were over-called as defects, and some smooth true
    # defects were missed -> a SYSTEMATIC error (not just random), which a model will learn.
    label_recorded = true_defect.copy()
    ra_hi = np.quantile(surface_ra, 0.60); ra_lo = np.quantile(surface_ra, 0.40)
    rough_ok = (true_defect == 0) & (surface_ra > ra_hi)
    label_recorded[rough_ok & (rng.random(N) < 0.50)] = 1          # rough good -> "defect"
    smooth_def = (true_defect == 1) & (surface_ra < ra_lo)
    label_recorded[smooth_def & (rng.random(N) < 0.50)] = 0        # smooth defect -> missed
    rand_flip = rng.random(N) < 0.06                               # light random noise on top
    label_recorded[rand_flip] = 1 - label_recorded[rand_flip]

    df = pd.DataFrame({
        "porosity_pct": porosity.round(2), "wall_thickness_mm": wall.round(2),
        "fill_time_s": fill_time.round(2), "melt_temp_c": melt_temp.round(1),
        "pressure_bar": pressure.round(1), "surface_ra_um": surface_ra.round(2),
        "inspector_A": insp_A, "inspector_B": insp_B, "inspector_C": insp_C,
        "label_recorded": label_recorded, "true_defect": true_defect,
    })
    df.to_csv(csv_path, index=False)
    if verbose:
        from itertools import combinations
        print("castings:", df.shape)
        print("true defect rate:", round(true_defect.mean(), 3))
        print("recorded-label error rate vs truth:", round((df.label_recorded != df.true_defect).mean(), 3))
        for a, b in combinations(["inspector_A", "inspector_B", "inspector_C"], 2):
            agree = (df[a] == df[b]).mean()
            print(f"  raw agreement {a[-1]}-{b[-1]}: {agree:.3f}")
        cons = (df[["inspector_A", "inspector_B", "inspector_C"]].sum(axis=1) >= 2).astype(int)
        print("  consensus(majority) error rate:", round((cons != df.true_defect).mean(), 3))
    return df

if not os.path.exists('casting_inspection.csv'):
    build_castings(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')

Generated dataset file.


In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
df = pd.read_csv('casting_inspection.csv')
feat_cols = ['porosity_pct', 'wall_thickness_mm', 'fill_time_s', 'melt_temp_c', 'pressure_bar', 'surface_ra_um']
print('rows:', df.shape)
print('recorded-label defect rate:', round(df.label_recorded.mean(), 3))
print('(true_defect is hidden ground truth — used only to measure our progress)')
df.head(3)

rows: (2400, 11)
recorded-label defect rate: 0.367
(true_defect is hidden ground truth — used only to measure our progress)


,porosity_pct,wall_thickness_mm,fill_time_s,melt_temp_c,pressure_bar,surface_ra_um,inspector_A,inspector_B,inspector_C,label_recorded,true_defect
0,1.28,5.11,2.73,710.8,103.7,2.08,0,0,1,0,0
1,1.14,5.47,2.37,706.4,85.7,2.84,1,0,0,1,0
2,0.58,6.69,2.16,709.9,92.9,3.90,0,0,0,0,0


#1. How much do the inspectors agree?

In [3]:
# -----------------------------------------------------------
# 🔹 1A. COHEN'S KAPPA BETWEEN EACH PAIR OF INSPECTORS
# -----------------------------------------------------------
from sklearn.metrics import cohen_kappa_score
pairs = [('inspector_A', 'inspector_B'), ('inspector_A', 'inspector_C'), ('inspector_B', 'inspector_C')]
for a, b in pairs:
    k = cohen_kappa_score(df[a], df[b])
    raw = (df[a] == df[b]).mean()
    print(f'{a[-1]}-{b[-1]}: raw agreement {raw:.3f} | Cohen kappa {k:.3f}')
print('\nKappa corrects raw agreement for chance. <0.4 weak, 0.4-0.6 moderate, 0.6-0.8 substantial.')

A-B: raw agreement 0.887 | Cohen kappa 0.723
A-C: raw agreement 0.855 | Cohen kappa 0.648
B-C: raw agreement 0.837 | Cohen kappa 0.608

Kappa corrects raw agreement for chance. <0.4 weak, 0.4-0.6 moderate, 0.6-0.8 substantial.


#2. Consensus labels beat any single annotator

In [4]:
# -----------------------------------------------------------
# 🔹 2A. MAJORITY VOTE ACROSS THE THREE INSPECTORS
# -----------------------------------------------------------
votes = df[['inspector_A', 'inspector_B', 'inspector_C']].sum(axis=1)
df['label_consensus'] = (votes >= 2).astype(int)   # >=2 of 3 say defect
acc_single = (df['label_recorded'] == df['true_defect']).mean()
acc_consensus = (df['label_consensus'] == df['true_defect']).mean()
print(f'single recorded label accuracy vs truth: {acc_single:.3f}')
print(f'majority-vote consensus accuracy vs truth: {acc_consensus:.3f}')
print('Aggregating independent noisy labels cancels random mistakes -> cleaner labels.')

single recorded label accuracy vs truth: 0.777
majority-vote consensus accuracy vs truth: 0.979
Aggregating independent noisy labels cancels random mistakes -> cleaner labels.


In [5]:
# 1. rows fixed by consensus
# Count rows where label_recorded is wrong but label_consensus is right
consensus_fixed_rows = df[(df['label_recorded'] != df['true_defect']) & (df['label_consensus'] == df['true_defect'])].shape[0]
print(f'Number of rows where label_recorded was wrong but consensus was right: {consensus_fixed_rows}')

# 2. why consensus helps on borderline parts: ...
# Explanation:
# Majority voting helps most on borderline parts where individual inspectors might make mistakes,
# but two out of three inspectors (or more in a larger panel) still agree on the correct label.
# In such cases, the collective wisdom tends to override the individual error.
# This is because random errors tend to cancel each other out when aggregated.
# The cost, however, is significant: you must pay for multiple labels (in this case, three inspectors
# instead of one) for each data point, which increases annotation expenses considerably.

Number of rows where label_recorded was wrong but consensus was right: 522


#3. Find likely label errors — triage what to re-inspect

In [6]:
# -----------------------------------------------------------
# 🔹 3A. CONFIDENT-LEARNING-STYLE ERROR DETECTION
# Train a model with cross-val; rows it confidently contradicts are suspect.
# -----------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X = df[feat_cols].values
y_noisy = df['label_recorded'].values
clf = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=300, random_state=0))
proba = cross_val_predict(clf, X, y_noisy, cv=5, method='predict_proba')[:, 1]
# suspect = model is confident the label is the OTHER class
suspect = ((proba > 0.80) & (y_noisy == 0)) | ((proba < 0.20) & (y_noisy == 1))
really_wrong = (df['label_recorded'] != df['true_defect']).values
base_rate = really_wrong.mean()
precision = (suspect & really_wrong).sum() / max(suspect.sum(), 1)
print(f'flagged {int(suspect.sum())} rows as suspect.')
print(f'of the flagged rows, {precision:.1%} really were wrong  (vs {base_rate:.1%} base error rate).')
print('The detector concentrates errors -> use it to TRIAGE which parts to re-inspect, not to auto-fix.')

flagged 107 rows as suspect.
of the flagged rows, 48.6% really were wrong  (vs 22.3% base error rate).
The detector concentrates errors -> use it to TRIAGE which parts to re-inspect, not to auto-fix.


In [7]:
# 1. real errors found vs random expectation
real_errors_in_flagged = (suspect & really_wrong).sum()
num_flagged_rows = suspect.sum()
expected_random_errors = num_flagged_rows * base_rate

print(f'Real errors found in flagged set: {real_errors_in_flagged}')
print(f'Expected errors if {num_flagged_rows} rows were picked randomly: {expected_random_errors:.1f}')

lift = real_errors_in_flagged / expected_random_errors if expected_random_errors > 0 else np.inf
print(f'Detection lift: {lift:.1f}x')

# 2. why lift matters under a budget: ...
# Explanation:
# A detector with ~2x lift (or more) is valuable even at <100% precision because it significantly
# concentrates the true errors into a smaller, prioritized subset of data. For a limited re-inspection budget,
# this means that by inspecting only the flagged items, you can find a much higher proportion of real errors
# than by randomly selecting items for re-inspection. For example, if you have a budget to re-inspect
# 100 items, and a detector provides a 2x lift, you'll find twice as many real errors as you would
# by randomly picking 100 items. This allows you to fix critical data quality issues far more efficiently,
# maximizing the impact of your re-inspection efforts without needing to check everything.

Real errors found in flagged set: 52
Expected errors if 107 rows were picked randomly: 23.9
Detection lift: 2.2x


#4. The data-centric payoff — re-labeling beats re-modelling

In [12]:
# -----------------------------------------------------------
# 🔹 4A. SAME MODEL, NOISY (single) vs CONSENSUS (re-labelled) TRAINING DATA
# Always evaluate against the clean ground truth on a held-out set.
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.3, random_state=42, stratify=df['true_defect'])
def train_eval(train_labels, model=None):
    # Fix: Only instantiate default model if 'model' is not provided, to avoid premature __len__ call
    if model is None:
        model = RandomForestClassifier(n_estimators=300, random_state=0)
    m = make_pipeline(StandardScaler(), model)
    m.fit(X[tr], train_labels[tr])
    return f1_score(df['true_defect'].values[te], m.predict(X[te]))
f1_noisy = train_eval(df['label_recorded'].values)
f1_consensus = train_eval(df['label_consensus'].values)   # from majority vote in step 2
print(f'F1 (trained on NOISY single labels):     {f1_noisy:.3f}')
print(f'F1 (trained on CONSENSUS re-labelled):   {f1_consensus:.3f}')
print(f'data-centric gain from better labels:    {f1_consensus - f1_noisy:+.3f}')
print('Same model, same features — only the labels improved.')

F1 (trained on NOISY single labels):     0.504
F1 (trained on CONSENSUS re-labelled):   0.667
data-centric gain from better labels:    +0.162
Same model, same features — only the labels improved.


In [13]:
from sklearn.ensemble import RandomForestClassifier # Changed from GradientBoostingClassifier

# 1. best model you can find, trained on NOISY labels, vs f1_consensus
# Using a deeper/larger RandomForestClassifier as suggested by the prompt
f1_gbt_noisy = train_eval(df['label_recorded'].values, model=RandomForestClassifier(n_estimators=500, max_depth=10, random_state=0))
print(f'F1 (Larger RF trained on NOISY single labels): {f1_gbt_noisy:.3f}')
print(f'F1 (trained on CONSENSUS re-labelled):   {f1_consensus:.3f}')
print(f'Data-centric gain (from prev step):    {f1_consensus - f1_noisy:+.3f}')
print(f'Model-centric gain (Larger RF on noisy): {f1_gbt_noisy - f1_noisy:+.3f}')

# 2. did model-swapping beat re-labelling? ...
# Explanation:
# In this experiment, even with a more powerful model like a deeper/larger RandomForestClassifier,
# training on the noisy 'label_recorded' data did not match the F1 score achieved by simply re-labeling the data
# to 'label_consensus' and training the original RandomForestClassifier. The data-centric gain from better labels
# (+0.162) significantly outperformed the model-centric gain from switching to a larger RandomForestClassifier on noisy labels
# (e.g., if F1_larger_RF_noisy is 0.515, the gain would be 0.515 - 0.504 = +0.011, which is much smaller than +0.162).
# This demonstrates the core data-centric argument: improving label quality often yields far greater performance
# gains than solely focusing on model architecture changes when dealing with noisy data.

F1 (Larger RF trained on NOISY single labels): 0.521
F1 (trained on CONSENSUS re-labelled):   0.667
Data-centric gain (from prev step):    +0.162
Model-centric gain (Larger RF on noisy): +0.017


#5. Active learning & weak supervision

In [14]:
# -----------------------------------------------------------
# 🔹 5A. ACTIVE LEARNING — LABEL THE MOST UNCERTAIN PARTS NEXT
# -----------------------------------------------------------
# Simulate a small labelled budget: which unlabelled parts should we send to inspection?
rng = np.random.default_rng(0)
labelled = rng.choice(idx, 150, replace=False)
pool = np.setdiff1d(idx, labelled)
m = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, random_state=0))
m.fit(X[labelled], df['true_defect'].values[labelled])
pool_proba = m.predict_proba(X[pool])[:, 1]
uncertainty = 1 - np.abs(pool_proba - 0.5) * 2     # 1 = most uncertain (proba near 0.5)
next_to_label = pool[np.argsort(-uncertainty)[:20]]
print('20 most informative parts to label next (indices):')
print(next_to_label[:20])
print('Active learning spends the labelling budget where the model is least sure.')

20 most informative parts to label next (indices):
[1027  737 1721 1760 2131 1723 1862  960  626 1276  891   84 1998  435
 1333  398  855 1535  196 1551]
Active learning spends the labelling budget where the model is least sure.


In [15]:
# 1-2. write labelling functions, combine, measure vs truth

# Define 3 labelling functions (LFs)
lf1 = df['porosity_pct'] > 5.0      # High porosity -> defect
lf2 = df['wall_thickness_mm'] < 4.0 # Thin wall -> defect
lf3 = df['surface_ra_um'] > 4.5     # Rough surface -> defect

# Combine LFs using 'any rule fires' (OR logic) for the weak label
# You could also use a majority vote: (lf1 + lf2 + lf3) >= 2
weak_label = (lf1 | lf2 | lf3).astype(int)

# Compare weak label accuracy vs true_defect
acc_weak_label = (weak_label == df['true_defect']).mean()
print(f'Weak label accuracy vs true_defect: {acc_weak_label:.3f}')

# Compare to a single inspector (e.g., inspector_A)
acc_inspector_A = (df['inspector_A'] == df['true_defect']).mean()
print(f'Inspector A accuracy vs true_defect: {acc_inspector_A:.3f}')

# 3. rules vs humans: ...
# Explanation:
# Rules can beat humans in consistency and scalability. Once defined, they apply the same logic
# perfectly to all data points, eliminating human fatigue or subjective biases. They are also
# highly scalable, labeling vast datasets quickly and cheaply. This can be particularly useful
# for initial labeling passes or for identifying obvious defects with clear thresholds.
#
# However, rules often fail to capture subtle or complex cases that require nuanced human judgment
# or pattern recognition. They are brittle to variations not explicitly covered by the rules
# and can miss edge cases. Developing effective rules requires significant domain expertise and
# iterative refinement, and they may struggle to achieve the same high precision or recall
# as a well-trained human or a machine learning model on complex, non-linear relationships.

Weak label accuracy vs true_defect: 0.776
Inspector A accuracy vs true_defect: 0.955


#📘 Summary

| Technique | What it buys you |
| --------- | ---------------- |
| Cohen's kappa | quantifies annotator (dis)agreement |
| Consensus labels | cancels random annotator error |
| Error detection | finds likely-mislabelled rows to fix |
| Clean-then-train | better labels lift the model (no model change) |
| Active learning | spend the labelling budget where it matters |
| Weak supervision | label at scale with rules |

**Core lesson:** in the data-centric mindset, *the labels are part of the model*. Measuring and improving label quality is often the highest-leverage thing you can do — frequently beating a fancier algorithm.

**Next — U20:** once a model is trained on good data, is it **fair** and **explainable**?